# EasyNCO TSP Benchmark（初始化-only 版本）——逐数据 × 逐方法的显式命令

本 notebook 的目标：

- **完全通过 EasyNCO 平台的命令行**来做评测（`python EasyNCO/eval.py ...`），不在 notebook 里手写“调用模型/环境”的逻辑。
- 对以下数据集里所有 TSP 数据文件，逐个方法跑一遍：
  - `EasyNCO/data/datasets/cross_distribution/`（TSP 部分）
  - `EasyNCO/data/datasets/test_dataset_tsp_uniform/`
- **所有方法都只评估“初始化（initialization）”的解质量**：也就是只看“初始解/构造解”的 cost（以及 aug 后的 best cost），不进行任何“迭代改进（iteration）”。

为什么要做“初始化-only”？

- 你当前的研究关注点是“初始化器 vs 迭代器”的组合与对比；因此需要把不同方法的 **initialization** 放到同一口径下对齐。
- 另一个现实原因：很多 iteration 阶段会非常慢、或对规模/数据格式更敏感；先把 initialization-only 跑通，能更快产出一版稳定可复现的基线。

输出位置（你跑完后主要看这里）：

- 每个方法 × 每个数据文件：都会生成一个日志文件
  - `results/benchmarks/tsp_datasets/logs/<method>/<dataset_key>.log`
- 日志里通常能看到：
  - `Eval ==> ... Score: ... Augmented Score: ...`（每个 batch 的进度）
  - `Eval Done ==> ... Score Summary: Avg: ..., Avg Augmented: ...`（最终平均）

重要提醒：关于“数据路径”

- `EasyNCO/eval.py` 会把 `test_data_path=...` 当作 **相对 `EasyNCO/data/datasets/` 的路径**。
- 你在 prompt 里提到的数据在 `final/data/datasets/...`；如果你那边只有 final 的数据，请把它们 **拷贝或软链接**到 `EasyNCO/data/datasets/` 下（不改代码，仅整理数据路径）。

下面所有 cell 都是“显式 cmd”（无循环拼接）。你可以按需逐个运行，也可以一口气全部运行。


## 0. 运行前自检（非常重要）

Notebook 通常位于 `benchmarks/` 目录里；而 EasyNCO 的命令需要在仓库根目录执行。

为避免“在 notebook 里跑 bash 时 CWD 不对”，本 notebook 的每个 `%%bash` cell 都会：

1. `git rev-parse --show-toplevel` 自动定位仓库根目录
2. `cd` 到仓库根目录后再执行 `python EasyNCO/eval.py ...`

如果你是在没有 `.git` 的环境（比如只拷了部分目录）运行，`git rev-parse` 会失败；这种情况下请手动把 notebook 的工作目录切到仓库根目录再跑。


In [1]:
from pathlib import Path

repo = Path.cwd().resolve()
print('当前工作目录:', repo)
print('是否存在 EasyNCO/eval.py:', (repo / 'EasyNCO' / 'eval.py').exists())
print('数据根目录(应存在):', repo / 'EasyNCO' / 'data' / 'datasets')
print('预训练 ckpt 目录(应存在):', repo / 'EasyNCO' / 'pretrained')
print('初始化-only 迭代器(应存在):', repo / 'benchmarks' / 'easynco_init_only_iteration.py')


当前工作目录: /public/home/zhoucl/shiys/benchmarks
是否存在 EasyNCO/eval.py: False
数据根目录(应存在): /public/home/zhoucl/shiys/benchmarks/EasyNCO/data/datasets
预训练 ckpt 目录(应存在): /public/home/zhoucl/shiys/benchmarks/EasyNCO/pretrained
初始化-only 迭代器(应存在): /public/home/zhoucl/shiys/benchmarks/benchmarks/easynco_init_only_iteration.py


## 1. 我们如何做到“只跑 initialization，不跑 iteration”？

EasyNCO 的 `ARREINFORCELightning.test_step()` 里逻辑是固定的：

- 先 `initialization.run(...)`
- 再 `iteration.run(...)`

所以如果你想**完全禁止 iteration 改进**，最稳的方式是：

- 不改 EasyNCO 源码
- 通过 Hydra 覆盖（override）把 `settings.iteration._target_` 换成我们自己写的 `InitOnlyIteration`

本仓库已经新增了一个极小的 helper：

- `benchmarks/easynco_init_only_iteration.py` 里的 `InitOnlyIteration`

它做的事情非常简单：

- 不做任何改进步骤
- 从 `initialization_out` 里抽取/计算出两个 python `float`：
  - `no_aug_score`：不做数据增强时的平均 cost
  - `aug_score`：做数据增强后取 best 的平均 cost

这样 `EasyNCO/eval.py` 的整体评测流程完全不变，但“迭代改进”被我们替换为一个“只负责读初始化结果”的空操作。

注意：有些方法（DeepACO / DIFUSCO / T2T）在 EasyNCO 里 **真正产生 tour/cost 的逻辑主要放在 iteration 阶段**，初始化阶段只生成中间结果（例如 heatmap）或直接把 batch 原样传下去。

因此：在严格 initialization-only 的约束下，这些方法**无法输出可比的 cost**，本 notebook 会对它们标记为 SKIP，并在日志里写清原因。


## 2. 方法清单（已检查 EasyNCO/settings 与 EasyNCO/pretrained）

本 notebook 会包含这些“与 TSP 相关”的方法：

- **会实际运行（init-only 可得到 cost）**：`omni`, `lehd`, `elg`, `icam`, `dact`, `htsp`
- **会出现但默认 SKIP（init-only 无法得到 cost）**：`deepaco`, `difusco`, `t2t`

另外 EasyNCO 里还有 `pomo`, `am`, `pointerformer`, `matnet` 等 TSP 方法实现；但你当前仓库的 `EasyNCO/pretrained/` 没有对应 ckpt，且你要求“尽量用现成命令跑”，所以这里不默认加入它们的 per-dataset 运行 block。

如果你后续补齐这些方法的 ckpt，我可以把它们也补进 notebook，并继续保持 init-only 评测口径。


## 3. `python EasyNCO/eval.py ...` 参数详解（建议先读一遍）

下面每个方法的命令基本长得一样；你只需要改少数几个关键参数就能控制实验。

### (A) 与“跑哪个方法”相关

- `settings=<xxx_settings>`：选择一份方法模板（在 `EasyNCO/settings/<xxx_settings>.yaml`）。它决定 env/model/initialization/iteration/module/trainer 等默认配置。
- `model=<method_name>`：方法名（主要用于日志展示、以及少数方法内部的分支）。
- `settings.test_loader.model_dirpath=pretrained` + `settings.test_loader.model_filename=*.ckpt`：指定要加载的 checkpoint（相对 `EasyNCO/` 目录）。

### (B) 与“跑哪个数据”相关

- `test_data_path=<relative_path>`：注意这里必须是 **相对 `EasyNCO/data/datasets/` 的路径**。
- `problem=tsp`：指定问题类型。
- `scale=N`：告诉 env 当前问题规模。大多数数据加载器会根据数据文件自动推断 N，但 EasyNCO 仍会把 `${scale}` 传进 env/model；因此这里保持和数据一致是最稳的。

### (C) 与“解码策略 / 增强”相关

- `decoder_strategy=greedy`：构造式方法通常支持 `greedy`/`sampling` 两种（有些方法只用 greedy）。
- `++settings.env.aug_factor=K`：坐标增强的次数（例如 POMO-style 的 rotate/reflect）。日志里一般会同时报告：
  - `Score Summary: Avg`（no-aug）
  - `Score Summary: Avg Augmented`（aug 后 best）

### (D) 与“只跑初始化（init-only）”相关（本 notebook 的核心）

- `settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration`
  - 这行会把原本的 iteration 替换成一个“空操作 + 读取初始化分数”的实现。
  - 作用等价于：**完全不做局部搜索 / 不做迭代改进**，只评估 initialization 产生的初始解质量。
  - 这不会修改 EasyNCO 源码，只是 Hydra 的配置覆盖。

### (E) 与“算力/可复现”相关

- `CUDA_VISIBLE_DEVICES=0` + `cuda=[0]`：选择 GPU。两者都保留是为了减少环境差异导致的“看不到 GPU / 选错 GPU”。
- `seed=1234`：固定随机种子。
- `episodes=...`：评测多少个实例。建议先把它调小（例如 16/32）跑通，再做全量。
- `batch_size=...`：每个 batch 的实例数。N 很大时必须减小，否则容易 OOM。

### (F) 日志怎么看

- 每个 cell 会把输出写到 `results/benchmarks/tsp_datasets/logs/<method>/<dataset_key>.log`。
- 一般你只需要从 log 里抓这两行：
  - `Eval Done ==> ... Score Summary: Avg: ..., Avg Augmented: ...`
  - `Total Testing Time: ... seconds`

（可选）如果你后续希望把这些 log 自动汇总成一个 TSV/CSV，我也可以给你再加一个“只读日志、生成表格”的 notebook 区块。


---

## Dataset: `cross_distribution/tsp1000_explosion.pkl`

- 规模（scale / N）= **1000**
- episodes（测试实例数）= **2000**
- 文件格式 = `pkl(list)`
- 备注：cross_distribution：通常用于分布外泛化（rotation/explosion）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [2]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：cross_distribution/tsp1000_explosion.pkl
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/cross_distribution__tsp1000_explosion.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][init-only] omni on cross_distribution/tsp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_explosion.pkl \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/cross_distribution__tsp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


[run][init-only] omni on cross_distribution/tsp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)
python: can't open file 'EasyNCO/eval.py': [Errno 2] No such file or directory
[exit_code] 2


fatal: not a git repository (or any parent up to mount point /public/home)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：cross_distribution/tsp1000_explosion.pkl
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/cross_distribution__tsp1000_explosion.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][init-only] lehd on cross_distribution/tsp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_explosion.pkl \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/cross_distribution__tsp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：cross_distribution/tsp1000_explosion.pkl
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/cross_distribution__tsp1000_explosion.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][init-only] elg on cross_distribution/tsp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_explosion.pkl \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/cross_distribution__tsp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：cross_distribution/tsp1000_explosion.pkl
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/cross_distribution__tsp1000_explosion.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][init-only] icam on cross_distribution/tsp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_explosion.pkl \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/cross_distribution__tsp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；且你要求 init-only，其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/cross_distribution__tsp1000_explosion.pkl.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：cross_distribution/tsp1000_explosion.pkl
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/cross_distribution__tsp1000_explosion.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][init-only] htsp on cross_distribution/tsp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_explosion.pkl \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/cross_distribution__tsp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/cross_distribution__tsp1000_explosion.pkl.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/difusco/cross_distribution__tsp1000_explosion.pkl.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/t2t/cross_distribution__tsp1000_explosion.pkl.log"


---

## Dataset: `cross_distribution/tsp1000_rotation.pkl`

- 规模（scale / N）= **1000**
- episodes（测试实例数）= **2000**
- 文件格式 = `pkl(list)`
- 备注：cross_distribution：通常用于分布外泛化（rotation/explosion）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：cross_distribution/tsp1000_rotation.pkl
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/cross_distribution__tsp1000_rotation.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][init-only] omni on cross_distribution/tsp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_rotation.pkl \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/cross_distribution__tsp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：cross_distribution/tsp1000_rotation.pkl
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/cross_distribution__tsp1000_rotation.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][init-only] lehd on cross_distribution/tsp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_rotation.pkl \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/cross_distribution__tsp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：cross_distribution/tsp1000_rotation.pkl
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/cross_distribution__tsp1000_rotation.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][init-only] elg on cross_distribution/tsp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_rotation.pkl \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/cross_distribution__tsp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：cross_distribution/tsp1000_rotation.pkl
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/cross_distribution__tsp1000_rotation.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][init-only] icam on cross_distribution/tsp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_rotation.pkl \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/cross_distribution__tsp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；且你要求 init-only，其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/cross_distribution__tsp1000_rotation.pkl.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：cross_distribution/tsp1000_rotation.pkl
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/cross_distribution__tsp1000_rotation.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][init-only] htsp on cross_distribution/tsp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_rotation.pkl \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/cross_distribution__tsp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/cross_distribution__tsp1000_rotation.pkl.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/difusco/cross_distribution__tsp1000_rotation.pkl.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/t2t/cross_distribution__tsp1000_rotation.pkl.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt`

- 规模（scale / N）= **100**
- episodes（测试实例数）= **10000**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][init-only] omni on test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=50, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=50 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][init-only] lehd on test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=128, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=128 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][init-only] elg on test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=50, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=50 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][init-only] icam on test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=50, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=50 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：dact
# 数据：test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/dact_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "[run][init-only] dact on test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=128, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=dact \
  problem=tsp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt \
  settings=dact_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=dact_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=128 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "SKIP: HTSP 的 lower-level group_size=200；当 N<200 时经常 shape mismatch，init-only 基线这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt`

- 规模（scale / N）= **200**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][init-only] omni on test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt (scale=200, episodes=128, batch_size=16, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][init-only] lehd on test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt (scale=200, episodes=128, batch_size=64, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=64 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][init-only] elg on test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt (scale=200, episodes=128, batch_size=16, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][init-only] icam on test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt (scale=200, episodes=128, batch_size=16, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；且你要求 init-only，其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][init-only] htsp on test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt (scale=200, episodes=128, batch_size=64, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=64 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt`

- 规模（scale / N）= **500**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][init-only] omni on test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt (scale=500, episodes=128, batch_size=8, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=8 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][init-only] lehd on test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt (scale=500, episodes=128, batch_size=32, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=32 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][init-only] elg on test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt (scale=500, episodes=128, batch_size=8, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=8 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][init-only] icam on test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt (scale=500, episodes=128, batch_size=8, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=8 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；且你要求 init-only，其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][init-only] htsp on test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt (scale=500, episodes=128, batch_size=32, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=32 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt`

- 规模（scale / N）= **1000**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][init-only] omni on test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt (scale=1000, episodes=128, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][init-only] lehd on test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt (scale=1000, episodes=128, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][init-only] elg on test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt (scale=1000, episodes=128, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][init-only] icam on test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt (scale=1000, episodes=128, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；且你要求 init-only，其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][init-only] htsp on test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt (scale=1000, episodes=128, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt`

- 规模（scale / N）= **2000**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][init-only] omni on test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt (scale=2000, episodes=128, batch_size=2, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=2000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=2 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][init-only] lehd on test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt (scale=2000, episodes=128, batch_size=8, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=2000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=8 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][init-only] elg on test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt (scale=2000, episodes=128, batch_size=2, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=2000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=2 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][init-only] icam on test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt (scale=2000, episodes=128, batch_size=2, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=2000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=2 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；且你要求 init-only，其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][init-only] htsp on test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt (scale=2000, episodes=128, batch_size=8, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=2000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=8 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt`

- 规模（scale / N）= **3000**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][init-only] omni on test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt (scale=3000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=3000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][init-only] lehd on test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt (scale=3000, episodes=128, batch_size=4, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=3000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][init-only] elg on test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt (scale=3000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=3000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][init-only] icam on test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt (scale=3000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=3000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；且你要求 init-only，其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][init-only] htsp on test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt (scale=3000, episodes=128, batch_size=4, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=3000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt`

- 规模（scale / N）= **4000**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][init-only] omni on test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt (scale=4000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=4000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][init-only] lehd on test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt (scale=4000, episodes=128, batch_size=2, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=4000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=2 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][init-only] elg on test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt (scale=4000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=4000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][init-only] icam on test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt (scale=4000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=4000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；且你要求 init-only，其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][init-only] htsp on test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt (scale=4000, episodes=128, batch_size=2, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=4000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=2 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt`

- 规模（scale / N）= **5000**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][init-only] omni on test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt (scale=5000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=5000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][init-only] lehd on test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt (scale=5000, episodes=128, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=5000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][init-only] elg on test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt (scale=5000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=5000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][init-only] icam on test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt (scale=5000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=5000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；且你要求 init-only，其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][init-only] htsp on test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt (scale=5000, episodes=128, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=5000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt`

- 规模（scale / N）= **10000**
- episodes（测试实例数）= **16**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][init-only] omni on test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt (scale=10000, episodes=16, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=10000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=16 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][init-only] lehd on test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt (scale=10000, episodes=16, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=10000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=16 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][init-only] elg on test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt (scale=10000, episodes=16, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=10000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=16 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][init-only] icam on test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt (scale=10000, episodes=16, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=10000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=16 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；且你要求 init-only，其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt
# 评测口径：init-only（通过 settings.iteration._target_ 覆盖为 InitOnlyIteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][init-only] htsp on test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt (scale=10000, episodes=16, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=10000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  settings.iteration._target_=benchmarks.easynco_init_only_iteration.InitOnlyIteration \
  seed=1234 \
  cuda=[0] \
  episodes=16 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。你要求严格 init-only，因此这里跳过。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"
